In [ ]:

load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")
URL = "https://www.googleapis.com/youtube/v3/commentThreads"
KEEP = ["comment_id", "text", "like_count", "published_at"]

def fetch_comments(video_id, max_pages=1):
    rows, token = [], None
    for _ in range(max_pages):
        params = {"part": "snippet", "videoId": video_id, "maxResults": 100,
                  "textFormat": "plainText", "order": "relevance", "key": API_KEY}
        if token:
            params["pageToken"] = token
        r = requests.get(URL, params=params, timeout=10)
        if r.status_code != 200:
            reason = r.json().get("error", {}).get("errors", [{}])[0].get("reason")
            print(f"[실패] {video_id}: {r.status_code} {reason}")
            return pd.DataFrame(columns=KEEP + ["video_id", "collected_at"])
        data = r.json()
        for item in data.get("items", []):
            s = item["snippet"]["topLevelComment"]["snippet"]
            rows.append({"comment_id": item["id"], "text": s.get("textDisplay", ""),
                         "like_count": s.get("likeCount", 0), "published_at": s.get("publishedAt")})
        token = data.get("nextPageToken")
        if not token:
            break
    df = pd.DataFrame(rows, columns=KEEP)
    df["video_id"] = video_id
    df["collected_at"] = datetime.now(timezone.utc).isoformat()
    return df

In [ ]:
# 유튜브 주소에서 영상 ID를 뽑고, 키와ID를 점검한 뒤 댓글을 받아 저장

import re
from pathlib import Path

def to_video_id(url_or_id):
    m = re.search(r"(?:v=|youtu\.be/|shorts/)([A-Za-z0-9_-]{11})", url_or_id)
    return m.group(1) if m else url_or_id.strip()

VIDEO_URL = "https://www.youtube.com/shorts/EEIjzche400"
VIDEO_ID = to_video_id(VIDEO_URL)

assert API_KEY, "키를 못 읽었습니다 → .env 위치 확인 (notebooks 폴더면 load_dotenv('../.env'))"
assert re.fullmatch(r"[A-Za-z0-9_-]{11}", VIDEO_ID), f"영상 ID 형식이 아닙니다: {VIDEO_ID}"

df = fetch_comments(VIDEO_ID)
assert not any("author" in c.lower() for c in df.columns), "작성자 정보가 섞였습니다"
print(VIDEO_ID, "→", len(df), "개 수집")
display(df.head())

out_dir = Path("../data/test" if Path.cwd().name == "notebooks" else "data/test")
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / f"{VIDEO_ID}.csv", index=False, encoding="utf-8-sig")

In [ ]:
# case_videos.csv의 영상 주소마다 존재·댓글 허용·댓글 수를 API로 확인해 채워 넣음

import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

base = Path("..") if Path.cwd().name == "notebooks" else Path(".")
path = base / "collector" / "case_videos.csv"
vdf = pd.read_csv(path, encoding="utf-8-sig", dtype=str).fillna("")

def check_video(vid):
    r = requests.get("https://www.googleapis.com/youtube/v3/videos",
                     params={"part": "snippet,statistics", "id": vid, "key": API_KEY}, timeout=10).json()
    if not r.get("items"):
        return {"video_exists": "N"}
    it = r["items"][0]
    probe = requests.get(URL, params={"part": "id", "videoId": vid, "maxResults": 1, "key": API_KEY}, timeout=10)
    return {"video_exists": "Y", "title": it["snippet"]["title"], "channel": it["snippet"]["channelTitle"],
            "published_at": it["snippet"]["publishedAt"][:10],
            "comments_enabled": "Y" if probe.status_code == 200 else "N",
            "comment_count": it["statistics"].get("commentCount", "")}

for i, row in vdf[vdf["video_url"] != ""].iterrows():
    for k, v in check_video(to_video_id(row["video_url"])).items():
        vdf.at[i, k] = v
    vdf.at[i, "checked_at"] = datetime.now(timezone.utc).strftime("%Y-%m-%d")

vdf.to_csv(path, index=False, encoding="utf-8-sig")
display(vdf[vdf["video_url"] != ""][["case_id", "video_exists", "comments_enabled", "comment_count", "title"]])

In [ ]:
# docs.db에 저장된 열 이름과 답글 3개 확인 (작성자 정보가 없어야 함)

import sqlite3
con = sqlite3.connect("../store/docs.db")
print([r[1] for r in con.execute("PRAGMA table_info(youtube_comments)")])
for r in con.execute("SELECT is_owner, text FROM youtube_comments WHERE is_reply=1 LIMIT 3"):
    print(r)
con.close()

# 작성자 이름, 채널 아이디, 프로필 등 없음 (개인정보 규칙 충족)
# 추출된 답글 중 @사용자 (언급) 이 없어서 닉네임 치환을 확인하지는 못했음 -> 전체 수집 확인 필요

In [ ]:
# 사례별 최상위·답글·채널주인 수와, @가 남은 답글 확인

import sqlite3, pandas as pd
con = sqlite3.connect("../store/docs.db")
print(pd.read_sql("""SELECT case_id, SUM(is_reply=0) AS 최상위, SUM(is_reply) AS 답글,
                     SUM(is_owner) AS 채널주인 FROM youtube_comments GROUP BY case_id""", con))
for r in con.execute("SELECT text FROM youtube_comments WHERE is_reply=1 AND text LIKE '%@%' LIMIT 5"):
    print(r[0][:60])
con.close()

In [ ]:
# 사례별 최상위·답글·채널주인 수와, @가 남은 답글 확인

import sqlite3, pandas as pd
con = sqlite3.connect("../store/docs.db")
print(pd.read_sql("""SELECT case_id, SUM(is_reply=0) AS 최상위, SUM(is_reply) AS 답글,
                     SUM(is_owner) AS 채널주인 FROM youtube_comments GROUP BY case_id""", con))
for r in con.execute("SELECT text FROM youtube_comments WHERE is_reply=1 AND text LIKE '%@%' LIMIT 5"):
    print(r[0][:60])
con.close()

# case_08(우체국)은 30개 미만
# case_02(야나두)는 채널 주인 답글이 16개 - indexer에서 빠질 예정

In [ ]:
# 사례별로 정제 후 남을 최상위 댓글 수 어림잡기 (채널주인 제외 → 10자 이상 → 한국어)

import re, sqlite3, pandas as pd
con = sqlite3.connect("../store/docs.db")
df = pd.read_sql("SELECT case_id, text FROM youtube_comments WHERE is_reply=0 AND is_owner=0", con)
con.close()

def ko_ratio(t):
    letters = re.findall(r"[A-Za-z가-힣\u3040-\u30ff\u4e00-\u9fff]", t)
    return sum("가" <= c <= "힣" for c in letters) / len(letters) if letters else 0

df["len_ok"] = df["text"].str.replace(r"\s", "", regex=True).str.len() >= 10
df["ko_ok"] = df["len_ok"] & (df["text"].map(ko_ratio) >= 0.5)
print(df.groupby("case_id").agg(최상위=("text", "size"), 열자이상=("len_ok", "sum"), 한국어=("ko_ok", "sum")))

In [ ]:
# case_08: 우체국 달달하이편 use=N, 종합편 추가, 공개 여부 변경, DB에서 달달하이편 댓글 삭제

import sqlite3, pandas as pd

V, C = "../collector/case_videos.csv", "../collector/cases.csv"
videos = pd.read_csv(V, encoding="utf-8-sig", dtype=str)
cases = pd.read_csv(C, encoding="utf-8-sig", dtype=str)

old = videos["video_url"].str.contains("myjoxHG6Puo")
videos.loc[old, "use"] = "N"
videos.loc[old, "note"] = "종합편으로 교체(2026-11-04). 영상 내 AI 표기 있음 → 종합편(표기 없음)과 공개 여부가 달라 섞지 않음"

if not videos["video_url"].str.contains("wlz7aUiz6WY").any():
    videos = pd.concat([videos, pd.DataFrame([{
        "case_id": "case_08", "video_url": "https://www.youtube.com/watch?v=wlz7aUiz6WY",
        "video_kind": "브랜드 공식", "title": "우체국예금 CF 출연은 처음이라... [종합편]",
        "channel": "우체국", "published_at": "2025-07-22", "video_exists": "Y",
        "comments_enabled": "Y", "comment_count": "360", "checked_at": "2026-11-04", "use": "Y",
        "note": "달달하이편과 같은 AI 제비 장면 포함. 영상·설명란·고정 댓글에 AI 표기 없음"}])],
        ignore_index=True)

c8 = cases["case_id"] == "case_08"
cases.loc[c8, "disclosed"] = "영상 내 표기 없음"
cases.loc[c8, "note"] = ("'보답하는 제비'. AI 사용 근거: 달달하이편 영상 내 AI 표기 + 종합편에 같은 제비 장면(2026-11-04 확인). "
                         "종합편은 영상·설명란·고정 댓글에 AI 표기 없음. 노컷뉴스 수상 보도는 직접 미확인")

videos.to_csv(V, index=False, encoding="utf-8-sig")
cases.to_csv(C, index=False, encoding="utf-8-sig")

con = sqlite3.connect("../store/docs.db")
with con:
    n = con.execute("DELETE FROM youtube_comments WHERE video_id='myjoxHG6Puo'").rowcount
con.close()
print("DB에서 삭제한 달달하이편 댓글:", n)
print(videos[videos.case_id == "case_08"][["video_url", "use", "comment_count"]])
print(cases[c8][["case_id", "disclosed"]])

In [ ]:
# 사례별로 정제 후 남을 최상위 댓글 수 어림잡기 (우체국 광고 링크 변경 후 재실행)

import re, sqlite3, pandas as pd
con = sqlite3.connect("../store/docs.db")
df = pd.read_sql("SELECT case_id, text FROM youtube_comments WHERE is_reply=0 AND is_owner=0", con)
con.close()

def ko_ratio(t):
    letters = re.findall(r"[A-Za-z가-힣\u3040-\u30ff\u4e00-\u9fff]", t)
    return sum("가" <= c <= "힣" for c in letters) / len(letters) if letters else 0

df["len_ok"] = df["text"].str.replace(r"\s", "", regex=True).str.len() >= 10
df["ko_ok"] = df["len_ok"] & (df["text"].map(ko_ratio) >= 0.5)
print(df.groupby("case_id").agg(최상위=("text", "size"), 열자이상=("len_ok", "sum"), 한국어=("ko_ok", "sum")))

# [[case_08    26    23   23]]에서 [[case_08   201   158  155]]로 변경

In [ ]:
# 인간 검수 결과물(label_check.csv)의 빈칸·오타·짝 규칙 위반 찾기

import pandas as pd
df = pd.read_csv("../eval/label_check.csv", encoding="utf-8-sig")
D, R = {"긍정","부정","중립","무관"}, {f"R{i}" for i in range(9)}
pair_ok = lambda d, r: (r in {"R1","R2","R3","R4","R5"} and d=="부정") or (r in {"R6","R7"} and d=="긍정") \
                       or (r in {"R0","R8"} and d!="무관") or (d=="무관" and r=="R0")
bad = df[~df.human_direction.isin(D) | ~df.human_reason.isin(R)
         | ~df.apply(lambda x: pair_ok(x.human_direction, x.human_reason), axis=1)]
print(f"문제 {len(bad)}개"); print(bad[["sample_id","human_direction","human_reason"]])

In [ ]:
# clean.py 정제 후 이벤트 필터가 제대로 걸렀는지 점검 (검수 메모와 비교, 읽기만 함)

import sqlite3, pandas as pd
con = sqlite3.connect("../store/docs.db")
c = pd.read_sql("SELECT y.comment_id, y.case_id, y.text, f.exclude_reason FROM youtube_comments y JOIN comment_filter f USING(comment_id)", con)
lab = pd.read_csv("../eval/label_check.csv", encoding="utf-8-sig").merge(c[["comment_id", "exclude_reason"]], on="comment_id")

print("검수 표본 중 정제에서 빠진 것\n", lab[lab.exclude_reason != ""][["sample_id", "case_id", "exclude_reason", "memo"]].to_string(), "\n")
print("이벤트의심 메모 댓글의 처리 결과\n", lab[lab.memo.fillna("").str.contains("이벤트")][["sample_id", "case_id", "exclude_reason", "text"]].to_string(), "\n")
print("event로 걸린 댓글\n", c[c.exclude_reason == "event"][["case_id", "text"]].to_string(), "\n")
ts = c[(c.exclude_reason == "") & c.text.str.match(r"\s*\d{1,2}:\d{2}")]
print("시간으로 시작하는데 남은 댓글 수\n", ts.case_id.value_counts().to_string())
print(ts.groupby("case_id").head(3)[["case_id", "text"]].to_string())

# event성 댓글로 걸린 5개는 모두 이벤트 응모 댓글이 아님을 확인
# 알바로 의심되는 댓글 (댓글마다 자주 보이는 타임스탬프)도 필터로 걸었지만, 댓글 하나하나 확인 결과 알바가 아닌 것으로 판정

# -> clean.py 수정
# "이벤트·응모" 같은 단어만으로 거르는 규칙 제거
# "타임스탬프로 시작 + 응원으로 끝남"만 필터 조건으로 남김

In [ ]:
# 빠진 검수 표본 4개의 원문과 짝이 된 중복 댓글 보기

import re, sqlite3, pandas as pd
con = sqlite3.connect("../store/docs.db")
c = pd.read_sql("SELECT y.comment_id, y.case_id, y.is_reply, y.like_count, y.text, f.exclude_reason FROM youtube_comments y JOIN comment_filter f USING(comment_id)", con)
key = lambda t: re.sub(r"[^가-힣A-Za-z0-9]|[ㅋㅎㅠㅜ]+", "", re.sub(r"[\u200b-\u200f\u2060\ufeff\u00ad]", "", t.replace("@사용자", "")))
c["key"] = c.text.fillna("").map(key)
lab = pd.read_csv("../eval/label_check.csv", encoding="utf-8-sig")
for sid in ["L017", "L061", "L084", "L092"]:
    r = c[c.comment_id == lab.loc[lab.sample_id == sid, "comment_id"].iloc[0]].iloc[0]
    print(f"\n[{sid}] {r.exclude_reason} / 답글={r.is_reply} / 좋아요={r.like_count}\n  원문: {r.text!r}")
    if r.exclude_reason == "dup":
        print(c[(c.case_id == r.case_id) & (c.key == r.key) & (c.comment_id != r.comment_id)][["is_reply", "like_count", "exclude_reason", "text"]].to_string())


# L017·L092 -> "(댓글 원문 비공개)" 반복 -> 빠져도 됨
# L061 -> "(댓글 원문 비공개)"(답글)와 "빙그레 참 잘했어요!~~~💖" -> 중복일 때 최상위 댓글을 먼저 남기기
# L084 -> “😅ใจใน😅" 이모지와 태국 문자 뿐.. -> 빠짐

In [ ]:
# 방향이 어떻게 엇갈렸는지 + R8↔R0 혼동 댓글 원문 보기

import pandas as pd
m = pd.read_csv("../eval/results/base_label_miss.csv", encoding="utf-8-sig")
print(pd.crosstab(m.human_direction, m.direction, rownames=["사람"], colnames=["GPT"]), "\n")
for _, r in m[m.human_reason.isin(["R8", "R0"]) & m.reason.isin(["R8", "R0"])].iterrows():
    print(f"[{r.sample_id}] 사람 {r.human_direction}/{r.human_reason} · GPT {r.direction}/{r.reason}\n   {r.text[:80]!r}")


# GPT가 광고 내용을 모르기 때문에 인간 예측과 다른 부분이 많았음
# 예) L004 "(댓글 원문 비공개)" / L005 "(댓글 원문 비공개)" / L050 "(댓글 원문 비공개)" (광고 속 소재와 ai 인물 이야기)
# 96개로 변수 하나씩 두 번 실험(exp1_codebook 규칙 수정 → exp1b_casedesc 광고 내용 추가)하고
# 최종 일치율은 규칙을 맞추는 데 쓰지 않은 새 40개로 재는 방식으로 변경

In [ ]:
# label_check2.csv(새 검수 40개) 형식 검사: 빈 칸, 허용 값, 방향·이유 짝 규칙

import pandas as pd
s = pd.read_csv("../eval/label_check2.csv", encoding="utf-8-sig", dtype=str)
d, r = s.human_direction.str.strip(), s.human_reason.str.strip().str.upper()
bad_pair = ((r.isin(["R1","R2","R3","R4","R5"]) & (d != "부정")) | (r.isin(["R6","R7"]) & (d != "긍정"))
            | ((d == "무관") & (r != "R0")))
problems = s[d.isna() | r.isna() | ~d.isin(["긍정","부정","중립","무관"])
             | ~r.isin([f"R{i}" for i in range(9)]) | bad_pair]
print(f"검수 {len(s)}개, 문제 {len(problems)}개")
print(problems[["sample_id", "human_direction", "human_reason", "memo"]].to_string())
print(d.value_counts().to_string(), "\n", r.value_counts().to_string())

In [ ]:
# 사례별 '무관 아닌 최상위 댓글' 수로 사례 자격 판정 (30개 이상) + 축소 기준 확인

import sqlite3, pandas as pd
con = sqlite3.connect("../store/docs.db")
d = pd.read_sql("""SELECT y.case_id, y.is_reply, l.direction FROM youtube_comments y
                   JOIN comment_label l USING(comment_id) WHERE l.run_tag = 'exp1b_casedesc'""", con)
top = d[d.is_reply == 0]
t = pd.crosstab(top.case_id, top.direction)
t["무관 아님"] = t.drop(columns="무관", errors="ignore").sum(axis=1)
t["부정 비율%"] = (t.get("부정", 0) / t["무관 아님"] * 100).round(1)
t["자격"] = t["무관 아님"].ge(30).map({True: "O", False: "X (탈락)"})
print(t.to_string())
n_ok = (t["무관 아님"] >= 30).sum()
print(f"\n자격 있는 사례 {n_ok}개 →", "축소 기준 해당 ('관찰됨' 등급까지만)" if n_ok < 10 else "축소 기준 해당 없음")

# 03·04·05·11이 탈락 - 7개 사례만 살아남음
# 자격 있는 사례가 10개 미만 -> 예상 등급은 '관찰됨'과 '근거 부족'만 운영
# 탈락 사례는 지식베이스에 남기되, '표본 부족, 예상 근거 제외'로 표시

In [ ]:
# S01 사례 정답 채우기: 자격 사례 중 R5 최상위 댓글 5개 이상 (기준은 숫자를 보기 전에 정함)

import json, sqlite3, pandas as pd
con = sqlite3.connect("../store/docs.db")
d = pd.read_sql("""SELECT y.case_id, COUNT(*) n FROM youtube_comments y JOIN comment_label l USING(comment_id)
                   JOIN case_summary s ON s.case_id = y.case_id
                   WHERE l.run_tag='exp1b_casedesc' AND y.is_reply=0 AND l.reason='R5' AND s.qualified=1
                   GROUP BY y.case_id""", con)
print(d.to_string(index=False))
ids = sorted(d[d.n >= 5].case_id)
p = "../eval/eval_set.jsonl"
rows = [json.loads(l) for l in open(p, encoding="utf-8")]
for r in rows:
    if r["id"] == "S01": r["gold_case_ids"] = ids
open(p, "w", encoding="utf-8").write("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows))
print("S01 정답 사례:", ids or "없음 → 근거가 적다는 안내 + 정부 기준이 정답")

In [ ]:
# 사례 요약 CSV에서 사례별 R5 수(최상위 부정 댓글 기준)와 자격 여부를 확인

import pandas as pd
cs = pd.read_csv("../report/case_summary.csv", encoding="utf-8-sig")
print(cs.columns.tolist())
cols = [c for c in cs.columns if "R5" in c or c in ("case_id", "qualified")]
print(cs[cols].to_string(index=False))

In [ ]:
# 사례별 최상위 부정 댓글 중 R5(창작자·저작권 우려) 개수와 비율 집계 (답글은 참고용으로 따로)

import sqlite3, pandas as pd
with sqlite3.connect("../store/docs.db") as con:
    g = pd.read_sql("""SELECT y.case_id, y.is_reply, l.direction, l.reason FROM youtube_comments y
                       JOIN comment_label l USING(comment_id) WHERE l.run_tag = 'exp1b_casedesc'""", con)
neg = g[g.direction == "부정"]
t = neg.groupby(["case_id", "is_reply"]).agg(neg_n=("reason", "size"), r5=("reason", lambda s: (s == "R5").sum())).unstack(fill_value=0)
t.columns = [f"{a}_{'답글' if b else '최상위'}" for a, b in t.columns]
t["r5_pct_최상위"] = (t["r5_최상위"] / t["neg_n_최상위"] * 100).round(1)
print(t.reindex(sorted(g.case_id.unique()), fill_value=0).to_string())

# 부정 이유 R5가 어떤 사례든 주된 이유로 나오지 않는다.

In [ ]:
# eval_set.jsonl의 S01 정답 사례를 01·02·10으로 채우고 answer_key의 안내 문구를 확정 메모로 교체

import json
p = "../eval/eval_set.jsonl"
rows = [json.loads(l) for l in open(p, encoding="utf-8")]
for r in rows:
    if r["id"] == "S01":
        r["gold_case_ids"] = ["case_01", "case_02", "case_10"]
        r["answer_key"] = r["answer_key"].replace(
            "gold_case_ids는 사례 요약 확정 후 R5가 집계된 사례로 채움",
            "gold_case_ids = R5가 최상위 부정 댓글에 1개 이상 나온 자격 사례(exp1b_casedesc): "
            "01 19개(13.4%), 10 4개(4.2%), 02 1개(11.1%, 근거 약함). R5는 어느 사례에서도 주된 이유가 아니므로 비율이 낮다고 말해야 정답")
        print(r)
with open(p, "w", encoding="utf-8") as f:
    f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)